URL index: [Home](https://zzz.bwh.harvard.edu/luna-walkthrough/) | [Data](https://zzz.bwh.harvard.edu/luna-walkthrough/data/) | [S1. File QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p1/) | [S2. Signal QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p2) | [S3. Staging](https://zzz.bwh.harvard.edu/luna-walkthrough/p3) | [S4. Artifacts](https://zzz.bwh.harvard.edu/luna-walkthrough/p4) | [S5. Analysis](https://zzz.bwh.harvard.edu/luna-walkthrough/p5)

Notebook index: [Index](../00_index.ipynb) | [S1. File QC](../p1/00_index.ipynb) | [S2. Signal QC](../p2/00_index.ipynb) | [S3. Staging](../p3/00_index.ipynb) | [S4. Artifacts](../p4/00_index.ipynb) | [S5. Analysis](../p5/00_index.ipynb)

---

# 4.1. Revising the current QC+ project

Walkthrough URL = [https://zzz.bwh.harvard.edu/luna-walkthrough/p4/harm2/](https://zzz.bwh.harvard.edu/luna-walkthrough/p4/harm2/)

In [4]:
import lunapi as lp
proj = lp.proj()
proj.sample_list( '../harm1.lst' )

initiated lunapi v1.2.3 <lunapi.lunapi0.luna object at 0x11c4174b0> 

read 20 individuals from ../harm1.lst


We've already handled the issues detected with `F01`, `F10` and `M10` (gapped EDFs) and `F05` (pre-filtered), and generated linked-mastoid, resampled versions in the `harm1.lst` project. Here we'll create a new project (`harm2`), address the remaining issues, and then perform a final step of epoch-level artifact detection and interpolation.

## Signals

We'll start by copying all `harm1` EDFs and then update the subset that need specific changes (from the `harm1` to the `harm2` set). Assuming you've created the `../work/harm2/` folder as required above:

In [3]:
import os
import shutil

src = '../work/harm1'
dst = '../work/harm2'
ext = '.edf'

os.makedirs( '../work/harm2' , exist_ok=True )

for file in os.listdir(src):
    if file.endswith(ext):
        shutil.copy2(os.path.join(src, file), os.path.join(dst, file))


Studies with known issues remaining:

 - `F04`: wrong units
 - `F07`, `F09`: flipped EEG polarity
 - `M01`: flat (midlines)
 - `M03`: dupes (CZ C3 C4 F3 F4 P3 P4)
 - `M04`: dropped (CZ)
 - various: varying levels of artifact based on the Hjorth signal reviews


### Rescaling units

In [5]:
# luna harm1.lst id=F04 -s ' SET-HEADERS unit=mV & uV & WRITE edf-dir=work/harm2 ' 
p = proj.inst( 'F04' ) 
p.eval( ' SET-HEADERS unit=mV & uV & WRITE edf-dir=../work/harm2 ' ) 

___________________________________________________________________
Processing: F04 | ../work/harm1/F04.edf
 duration 07.18.42, 26322s | time 22.00.00 - 05.18.42 | date 01.01.85

 signals: 57 (of 57) selected in a standard EDF file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ
 ..................................................................
 CMD #1: SET-HEADERS
   options: sig=* unit=mV
  set Fp1 'unit' to mV
  set Fp2 'unit' to mV
  set AF3 'unit' to mV
  set AF4 'unit' to mV
  set F7 'unit' to mV
  set F5 'unit' to mV
  set F3 'unit' to mV
  set F1 'unit' to mV
  set F2 'unit' to mV
  set F4 'unit' to mV
  set F6 'unit' to mV
  set F8 'unit' to mV
  set FT7 'unit' to mV
  set FC5 'unit' to mV
  set FC3 'unit' to mV
  se

,Command,Strata
0,WRITE,BL


### Flipping EEGs

In [6]:
# luna harm1.lst id=F07,F09 -s ' FLIP & WRITE edf-dir=work/harm2 ' 
p = proj.inst( 'F07' ) 
p.eval( ' FLIP & WRITE edf-dir=../work/harm2 ' )

p = proj.inst( 'F09' ) 
p.eval( ' FLIP & WRITE edf-dir=../work/harm2 ' )

___________________________________________________________________
Processing: F07 | ../work/harm1/F07.edf
 duration 07.52.06, 28326s | time 22.00.00 - 05.52.06 | date 01.01.85

 signals: 57 (of 57) selected in a standard EDF file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ
 ..................................................................
 CMD #1: FLIP
   options: sig=*
  flipping polarity of Fp1
  flipping polarity of Fp2
  flipping polarity of AF3
  flipping polarity of AF4
  flipping polarity of F7
  flipping polarity of F5
  flipping polarity of F3
  flipping polarity of F1
  flipping polarity of F2
  flipping polarity of F4
  flipping polarity of F6
  flipping polarity of F8
  flipping polarity of FT7
  flipping po

,Command,Strata
0,FLIP,CH
1,WRITE,BL


flipping polarity of Fp2
  flipping polarity of AF3
  flipping polarity of AF4
  flipping polarity of F7
  flipping polarity of F5
  flipping polarity of F3
  flipping polarity of F1
  flipping polarity of F2
  flipping polarity of F4
  flipping polarity of F6
  flipping polarity of F8
  flipping polarity of FT7
  flipping polarity of FC5
  flipping polarity of FC3
  flipping polarity of FC1
  flipping polarity of FC2
  flipping polarity of FC4
  flipping polarity of FC6
  flipping polarity of FT8
  flipping polarity of T7
  flipping polarity of C5
  flipping polarity of C3
  flipping polarity of C1
  flipping polarity of C2
  flipping polarity of C4
  flipping polarity of C6
  flipping polarity of T8
  flipping polarity of TP7
  flipping polarity of CP5
  flipping polarity of CP3
  flipping polarity of CP1
  flipping polarity of CP2
  flipping polarity of CP4
  flipping polarity of CP6
  flipping polarity of TP8
  flipping polarity of P7
  flipping polarity of P5
  flipping polarity o

### Adding CZ for M04

In [8]:
# luna harm1.lst id=M04 \
#  -s ' TRANS sig=CZ expr=" CZ = C1 "
#       WRITE edf-dir=work/harm2 '

p = proj.inst( 'M04' ) 
p.eval( ''' TRANS sig=CZ expr=" CZ = C1 "
            WRITE edf-dir=../work/harm2 ''' ) 

___________________________________________________________________
Processing: M04 | ../work/harm1/M04.edf
 duration 07.55.32, 28532s | time 22.00.00 - 05.55.32 | date 01.01.85

 signals: 56 (of 56) selected in a standard EDF file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CPZ | PZ | POz | OZ | FPZ
 ..................................................................
 CMD #1: TRANS
   options: expr=" CZ = C1 " sig=CZ
  evaluating expression  :  CZ = C1  ; CZ
  attaching C1 for 3652096 sample-points...
  returned 3652096 sample-points
  creating new channel CZ...
 ..................................................................
 CMD #2: WRITE
   options: edf-dir=../work/harm2 sig=*
  no epoch mask set, no restructuring needed
  data are not truly disco

,Command,Strata
0,WRITE,BL


## Annotations

In terms of annotations, we have detected all of the injected manipulations. Thus at this point we'll simply correct these by copying all `v1` (original) annotation files over, which fixes the issues of swapped, scrambled and truncated stage annotations. `POPS` staging effectively seemed to solve these issues also, but we'll ignore that for now and use the available manual staging in this walkthrough.

In [11]:
# cp luna-grins/v1/annots/* work/harm2/
shutil.copytree('../luna-grins/v1/annots/', '../work/harm2/', dirs_exist_ok = True )

'../work/harm2/'

## Sample list

We should now have populated `work/harm2/` with 20 EDFs and 20 matching .annot files.

 - 16 EDFs are copied directly from `work/harm1`, whereas 4 were updated (i.e. rescaled, flipped EEGs, channel added)

 - 20 annotation files have been copied from the original `v1` (unmanipulated) dataset (from `luna-grins/v1/annots/`)

For the pipeline in this walkthrough, the only remaining QC step is to apply epoch-level interpolation, as previously described here.

To that end, we'll build a new sample-list, on which we'll base all parts of this step:

In [12]:
# luna --build work/harm2 > harm2.lst
proj.build( '../work/harm2' )  

20

In [13]:
proj.sample_list()

,ID,EDF,Annotations
1,F01,../work/harm2/F01.edf,{../work/harm2/F01.annot}
2,F02,../work/harm2/F02.edf,{../work/harm2/F02.annot}
3,F03,../work/harm2/F03.edf,{../work/harm2/F03.annot}
4,F04,../work/harm2/F04.edf,{../work/harm2/F04.annot}
5,F05,../work/harm2/F05.edf,{../work/harm2/F05.annot}
6,F06,../work/harm2/F06.edf,{../work/harm2/F06.annot}
7,F07,../work/harm2/F07.edf,{../work/harm2/F07.annot}
8,F08,../work/harm2/F08.edf,{../work/harm2/F08.annot}
9,F09,../work/harm2/F09.edf,{../work/harm2/F09.annot}
10,F10,../work/harm2/F10.edf,{../work/harm2/F10.annot}


In [14]:
# save sample list to harm2.lst
sl = proj.sample_list()
sl['Annotations'] = sl['Annotations'].map( lambda x: str(x).strip('{}\'') )
sl.to_csv( '../harm2.lst', sep="\t", index=None, header=False)

---

We can now move on to [interpolation](02_interp.ipynb).